In [1]:
from pathlib import Path
import sys

# ---------------------------------------------------------------------
# Locate project root
# ---------------------------------------------------------------------

project_root = Path.cwd()

while project_root.name != "EventCameraProject":
    if project_root.parent == project_root:
        raise RuntimeError("Could not locate EventCameraProject root.")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project Root:", project_root)

Project Root: /home/ayon/git/EventCameraProject


In [2]:
def tensor_info(name, x):
    print(f"\n{name}")
    print("-" * 90)
    print("Shape :", tuple(x.shape))
    print("Dtype :", x.dtype)
    print("Device:", x.device)
    print("Mean  :", float(x.mean()))
    print("Std   :", float(x.std()))
    print("Min   :", float(x.min()))
    print("Max   :", float(x.max()))
    print("NaNs  :", bool(torch.isnan(x).any()))
    print("Infs  :", bool(torch.isinf(x).any()))

In [3]:
from pathlib import Path

import torch
from torch.utils.data import DataLoader

from src.data.dataset import EVIMO2Dataset
from src.data.temporal_dataset import TemporalEVIMO2Dataset
from src.data.collate import temporal_collate_fn

from src.data.transforms import (
    Compose,
    ToTensor,
    NormalizeEventTime,
    NormalizeIMU,
    VoxelizeEvents,
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cpu


/home/ayon/miniconda3/envs/EventProject/lib/python3.11/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [4]:
from pathlib import Path
import sys

# ---------------------------------------------------------------------
# Locate project root
# ---------------------------------------------------------------------

project_root = Path.cwd()

while project_root.name != "EventCameraProject":
    if project_root.parent == project_root:
        raise RuntimeError("Could not locate EventCameraProject root.")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project Root:", project_root)

# ---------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------

import torch
from torch.utils.data import DataLoader

from src.data.dataset import EVIMO2Dataset
from src.data.temporal_dataset import TemporalEVIMO2Dataset
from src.data.collate import temporal_collate_fn

from src.data.transforms import (
    Compose,
    ToTensor,
    NormalizeEventTime,
    NormalizeIMU,
    VoxelizeEvents,
)

# ---------------------------------------------------------------------
# Device
# ---------------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

# ---------------------------------------------------------------------
# Dataset
# ---------------------------------------------------------------------

dataset_root = Path(
    "/home/ayon/HDD/EventDatasets/EVIMO2_official"
)

frame_dataset = EVIMO2Dataset(
    dataset_root=dataset_root,
    sensors=("left_camera", "right_camera"),
    split="train",
    load_depth=True,
    load_mask=True,
)

temporal_dataset = TemporalEVIMO2Dataset(
    frame_dataset,
    history_offsets=(-3, -2, -1, 0),
)

loader = DataLoader(
    temporal_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=temporal_collate_fn,
)

# ---------------------------------------------------------------------
# Transform Pipeline
# ---------------------------------------------------------------------

transform = Compose(
    [
        ToTensor(),
        NormalizeEventTime(),
        NormalizeIMU(),
        VoxelizeEvents(
            num_bins=5,
        ),
    ]
)

# ---------------------------------------------------------------------
# Get one batch
# ---------------------------------------------------------------------

raw_batch = next(iter(loader))

voxel_batch = transform(raw_batch)

# ---------------------------------------------------------------------
# Build Event Tensor
# ---------------------------------------------------------------------

voxels = torch.stack(
    [
        frame.voxel_grid
        for frame in voxel_batch.frames
    ],
    dim=1,
).to(device)

print()
print("=" * 90)
print("EVENT VOXELS")
print("=" * 90)
print("Shape :", tuple(voxels.shape))
print("dtype :", voxels.dtype)
print("device:", voxels.device)

assert voxels.ndim == 5

batch_size, sequence_length, num_bins, height, width = voxels.shape

print()
print(f"Batch Size      : {batch_size}")
print(f"Sequence Length : {sequence_length}")
print(f"Voxel Bins      : {num_bins}")
print(f"Resolution      : {height} x {width}")

Project Root: /home/ayon/git/EventCameraProject
Device: cpu
EVIMO2 Sequence Index
Sequences : 22
Frames    : 9352
Sensors   : left_camera, right_camera
Split     : train

EVENT VOXELS
Shape : (2, 4, 5, 480, 640)
dtype : torch.float32
device: cpu

Batch Size      : 2
Sequence Length : 4
Voxel Bins      : 5
Resolution      : 480 x 640


In [14]:
import torch

from src.models.world_model.event_encoder import EventEncoder
from src.models.world_model.imu_encoder import IMUEncoder
from src.models.world_model.fusion import MotionFusion


from src.models.world_model.temporal_encoder import TemporalEncoder
from src.models.world_model.world_transition import WorldTransition
from src.models.world_model.pose_head import PoseHead
from src.models.world_model.depth_head import DepthHead
from src.models.world_model.latent_renderer import LatentRenderer
from src.models.world_model.alignment import Alignment
from src.models.world_model.temporal_memory import TemporalMemory
from src.models.world_model.decoder import WorldDecoder
from src.models.world_model.mask_head import DynamicMaskHead



device = "cuda" if torch.cuda.is_available() else "cpu"

event_encoder = EventEncoder(
    input_channels=num_bins,
).to(device)


imu_encoder = IMUEncoder(
    hidden_channels=64,
    embedding_dim=128,
).to(device)


motion_fusion = MotionFusion(
    event_channels=256,
    imu_dim=128,
).to(device)


temporal_encoder = TemporalEncoder(

    input_channels=256,

    hidden_channels=256,

    kernel_size=3,

).to(device)

pose_head = PoseHead(
    input_dim=128,
).to(device)

depth_head = DepthHead(
    input_channels=256,
).to(device)

renderer = LatentRenderer()

alignment = Alignment(
    channels=256,
)


In [9]:
event_features = event_encoder(voxels)

motion_embeddings = []

for temporal_index, frame in enumerate(voxel_batch.frames):

    embedding = imu_encoder(
        frame=frame,
        batch_size=batch_size,
    )

    motion_embeddings.append(embedding)

motion_embeddings = torch.stack(
    motion_embeddings,
    dim=1,
)

event_feature = event_features[-1]


fused = motion_fusion(
    event_feature,
    motion_embeddings,
)


temporal_features = temporal_encoder(
    fused
)



# previous_state = temporal_features[:, -2]

# target_state = temporal_features[:, -1]


imu_motion = motion_embeddings[:, -1]

pose = pose_head(
    imu_motion
)

latent_feature = temporal_features[:, -1]
print(latent_feature.shape)

depth = depth_head(
    latent_feature
)

K = torch.from_numpy(
    __import__("numpy").stack(frame.camera_intrinsics)
).float()

distortion = torch.from_numpy(
    __import__("numpy").stack(frame.camera_distortion)
).float()


print(depth.shape)

warped = renderer(
    latent_feature,
    depth,
    pose,
    K,
    distortion,
)

print(warped.shape)



torch.Size([2, 256, 30, 40])
torch.Size([2, 1, 30, 40])
torch.Size([2, 256, 30, 40])


In [ ]:
import torch
import torch.nn as nn

from src.models.world_model.event_encoder import EventEncoder
from src.models.world_model.imu_encoder import IMUEncoder
from src.models.world_model.fusion import MotionFusion


from src.models.world_model.temporal_encoder import TemporalEncoder
from src.models.world_model.world_transition import WorldTransition
from src.models.world_model.pose_head import PoseHead
from src.models.world_model.depth_head import DepthHead
from src.models.world_model.latent_renderer import LatentRenderer
from src.models.world_model.alignment import Alignment
from src.models.world_model.temporal_memory import TemporalMemory
from src.models.world_model.decoder import WorldDecoder
from src.models.world_model.mask_head import DynamicMaskHead


class WorldModel(nn.Module):

    def __init__(
        self,
        num_bins: int = 5,
        event_channels: int = 256,
        imu_hidden: int = 64,
        imu_embedding: int = 128,
        decoder_channels: int = 16,
    ):
        super().__init__()

        self.event_encoder = EventEncoder(
            input_channels=num_bins,
        )

        self.imu_encoder = IMUEncoder(
            hidden_channels=imu_hidden,
            embedding_dim=imu_embedding,
        )

        self.motion_fusion = MotionFusion(
            event_channels=event_channels,
            imu_dim=imu_embedding,
        )

        self.temporal_encoder = TemporalEncoder(
            input_channels=event_channels,
            hidden_channels=event_channels,
            kernel_size=3,
        )

        self.depth_head = DepthHead(
            input_channels=event_channels,
        )

        self.pose_head = PoseHead(
            input_dim=imu_embedding,
        )

        self.transition = WorldTransition(
            state_channels=event_channels,
            motion_dim=imu_embedding,
        )

        self.renderer = LatentRenderer()

        self.alignment = Alignment(
            channels=event_channels,
        )

        self.temporal_memory = TemporalMemory(
            channels=event_channels,
        )

        self.decoder = WorldDecoder(
            input_channels=event_channels,
            output_channels=decoder_channels,
        )

        self.mask_head = DynamicMaskHead(
            in_channels=decoder_channels,
        )


    def forward(self, voxel_batch, batch):
        event_pyramid = self.event_encoder(voxel_batch)

        event_features = event_pyramid[-1]

        motion_embeddings = []

        for frame in batch.frames:

            embedding = self.imu_encoder(
                frame=frame,
                batch_size=voxel_batch.shape[0],
            )

            motion_embeddings.append(embedding)

        motion_embeddings = torch.stack(
            motion_embeddings,
            dim=1,
        )

        fused = self.motion_fusion(
            event_features,
            motion_embeddings,
        )

        temporal_features = self.temporal_encoder(
            fused
        )

        current_feature = temporal_features[:, -1]

        depth = self.depth_head(
            current_feature
        )

        pose = self.pose_head(
            motion_embeddings[:, -1]
        )

        predicted_state = self.transition(
            current_feature,
            motion_embeddings[:, -1],
        )

        K = torch.stack([
            torch.as_tensor(k, device=voxel_batch.device)
            for k in batch.frames[-1].camera_intrinsics
        ])

        distortion = torch.stack([
            torch.as_tensor(d, device=voxel_batch.device)
            for d in batch.frames[-1].camera_distortion
        ])

        rendered = self.renderer(
            predicted_state,
            depth,
            pose,
            K,
            distortion,
        )

        aligned_input = temporal_features.clone()

        aligned_input[:, -1] = rendered

        aligned = self.alignment(
            aligned_input
        )

        world_feature = self.temporal_memory(
            aligned
        )

        decoded = self.decoder(
            world_feature
        )

        mask = self.mask_head(
            decoded
        )

        return {

            "event_features": event_features,

            "motion_embeddings": motion_embeddings,

            "fused_features": fused,

            "temporal_features": temporal_features,

            "depth": depth,

            "pose": pose,

            "predicted_state": predicted_state,

            "rendered_state": rendered,

            "aligned_features": aligned,

            "world_feature": world_feature,

            "decoded_feature": decoded,

            "mask": mask,
        }





In [12]:
model = WorldModel().to(device)
output = model(voxels, voxel_batch)


jij


In [19]:
model = WorldModel().to(device)

model.eval()
outputs = model(voxels, voxel_batch)

jij


In [20]:
print()
print("=" * 90)
print("WORLD MODEL INTEGRATION TEST")
print("=" * 90)

print()

for name, tensor in outputs.items():

    print(f"{name}")

    print("-" * 90)

    print("Shape :", tuple(tensor.shape))
    print("dtype :", tensor.dtype)
    print("device:", tensor.device)

    print(
        "Min   :",
        tensor.min().item(),
    )

    print(
        "Max   :",
        tensor.max().item(),
    )

    print(
        "Mean  :",
        tensor.mean().item(),
    )

    print(
        "Std   :",
        tensor.std().item(),
    )

    print(
        "NaNs  :",
        torch.isnan(tensor).any().item(),
    )

    print(
        "Infs  :",
        torch.isinf(tensor).any().item(),
    )

    assert torch.isfinite(tensor).all()

    print()


# ==========================================================
# Shape verification
# ==========================================================

B = batch_size

assert outputs["event_features"].shape == (
    B,
    4,
    256,
    30,
    40,
)

assert outputs["motion_embeddings"].shape == (
    B,
    4,
    128,
)

assert outputs["fused_features"].shape == (
    B,
    4,
    256,
    30,
    40,
)

assert outputs["temporal_features"].shape == (
    B,
    4,
    256,
    30,
    40,
)

assert outputs["depth"].shape == (
    B,
    1,
    30,
    40,
)

assert outputs["pose"].shape == (
    B,
    6,
)

assert outputs["predicted_state"].shape == (
    B,
    256,
    30,
    40,
)

assert outputs["rendered_state"].shape == (
    B,
    256,
    30,
    40,
)

assert outputs["aligned_features"].shape == (
    B,
    4,
    256,
    30,
    40,
)

assert outputs["world_feature"].shape == (
    B,
    256,
    30,
    40,
)

assert outputs["decoded_feature"].shape == (
    B,
    16,
    480,
    640,
)

assert outputs["mask"].shape == (
    B,
    1,
    480,
    640,
)

print("=" * 90)
print("ALL SHAPES VERIFIED")
print("=" * 90)


# ==========================================================
# Gradient test
# ==========================================================

loss = outputs["mask"].mean()

model.zero_grad()

loss.backward()

num_grad = 0

for name, parameter in model.named_parameters():

    if parameter.grad is not None:

        assert torch.isfinite(parameter.grad).all()

        num_grad += 1

print()

print("=" * 90)
print("GRADIENT CHECK")
print("=" * 90)

print(
    "Parameters with gradients:",
    num_grad,
)

assert num_grad > 0


# ==========================================================
# CUDA
# ==========================================================

if torch.cuda.is_available():

    print()

    print("=" * 90)
    print("CUDA MEMORY")
    print("=" * 90)

    print(
        f"Allocated : {torch.cuda.memory_allocated()/1024**2:.2f} MB"
    )

    print(
        f"Reserved  : {torch.cuda.memory_reserved()/1024**2:.2f} MB"
    )


print()

print("=" * 90)
print("✓ WORLD MODEL INTEGRATION TEST PASSED")
print("=" * 90)


WORLD MODEL INTEGRATION TEST

event_features
------------------------------------------------------------------------------------------
Shape : (2, 4, 256, 30, 40)
dtype : torch.float32
device: cpu
Min   : -0.27846455574035645
Max   : 30.769227981567383
Mean  : 0.3490944802761078
Std   : 1.1085952520370483
NaNs  : False
Infs  : False

motion_embeddings
------------------------------------------------------------------------------------------
Shape : (2, 4, 128)
dtype : torch.float32
device: cpu
Min   : -0.8547625541687012
Max   : 1.1352180242538452
Mean  : 0.029247891157865524
Std   : 0.3423348665237427
NaNs  : False
Infs  : False

fused_features
------------------------------------------------------------------------------------------
Shape : (2, 4, 256, 30, 40)
dtype : torch.float32
device: cpu
Min   : -0.27846455574035645
Max   : 15.299076080322266
Mean  : 0.14601543545722961
Std   : 0.6254635453224182
NaNs  : False
Infs  : False

temporal_features
---------------------------------

In [21]:
print(outputs["mask"].requires_grad)
print(outputs["mask"].grad_fn)

True
